In [38]:
import numpy as np
import os
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel
from scipy.optimize import minimize
from scipy.stats import norm
from pathlib import Path
import warnings

warnings.filterwarnings('ignore')

In [39]:
def ucb_acquisition(x, gp, Y_max, beta):
    #Upper Confidence Bound (UCB) acquisition function
    x = x.reshape(1, -1)

    # Get the mean (mu) and std (sigma) from the GP
    mu, sigma = gp.predict(x, return_std=True)
    
    # Calculate UCB: mu + sqrt(beta) * sigma
    ucb = mu + np.sqrt(beta) * sigma
    
    return -ucb[0] # Return the negative UCB for minimisation

In [40]:
submission_queries = {} # Dictionary to hold all 8 query strings
root = Path.cwd().parent.parent

base_directory = '..\..\data\initial_data'
beta_exploration = 2.0 #will increase for functions 6 to 8

for i in range(1, 9):
    print(f"\n--- Function {i} ---")
    
    folder_name = f'function_{i}'
    folder_path = os.path.join(root, "data/initial_data", folder_name)
    print(os.path.join(folder_path, 'initial_inputs.npy'))
    X = np.load(os.path.join(folder_path, 'initial_inputs.npy'))
    Y = np.load(os.path.join(folder_path, 'initial_outputs.npy'))
    
    # dimensionality
    dimension = X.shape[1]
    
    if Y.ndim > 1:
        Y = Y.flatten()
        
    Y_max = np.max(Y)

    
    print(f"{dimension}-Dimension - Y_max: {Y_max:.4f}")
    
    # 2. Build and Train GP
    kernel = Matern(length_scale=np.ones(dimension), length_scale_bounds=(1e-1, 1e2), nu=2.5)
    gp = GaussianProcessRegressor(
        kernel=kernel, alpha=1e-6, n_restarts_optimizer=20, random_state=42)
    gp.fit(X, Y)
    
    # 3. Define the Optimisation Task
    # Bounds for all functions are assumed to be [0.0, 1.0] for all dimensions.
    bounds = [(0.0, 1.0)] * dimension
    
    # Set the UCB exploration parameter based on dimensionality (Strategy: High Beta)
    # Increase Beta for higher dimensions where data is sparser.
    beta = beta_exploration
    if dimension >= 5:
        beta = beta_exploration * 1.5 # More aggressive exploration for high-D
        
    # The objective function (to be minimse) is the negative UCB
    ucb_objective = lambda x: ucb_acquisition(x, gp, Y_max, beta)
    
    # 4. Find the Argmax (i.e., minimise the negative UCB)
    # Use multiple random starts to avoid local minima in the acquisition function
    best_ucb_value = np.inf
    
    x_next = None
    
    for _ in range(20*dimension): # 20*D random restarts for better robustness
        x0 = np.random.uniform(0, 1, dimension)
        
        # Use a general minimiser (L-BFGS-B is good for bounded problems)
        res = minimize(ucb_objective, x0, bounds=bounds, method='L-BFGS-B')
        
        if res.fun < best_ucb_value:
            best_ucb_value = res.fun
            x_next = res.x



--- Function 1 ---
/workspaces/codespaces-jupyter/data/initial_data/function_1/initial_inputs.npy
2-Dimension - Y_max: 0.0000

--- Function 2 ---
/workspaces/codespaces-jupyter/data/initial_data/function_2/initial_inputs.npy
2-Dimension - Y_max: 0.6112

--- Function 3 ---
/workspaces/codespaces-jupyter/data/initial_data/function_3/initial_inputs.npy
3-Dimension - Y_max: -0.0348

--- Function 4 ---
/workspaces/codespaces-jupyter/data/initial_data/function_4/initial_inputs.npy
4-Dimension - Y_max: -4.0255

--- Function 5 ---
/workspaces/codespaces-jupyter/data/initial_data/function_5/initial_inputs.npy
4-Dimension - Y_max: 1088.8596

--- Function 6 ---
/workspaces/codespaces-jupyter/data/initial_data/function_6/initial_inputs.npy
5-Dimension - Y_max: -0.7143

--- Function 7 ---
/workspaces/codespaces-jupyter/data/initial_data/function_7/initial_inputs.npy
6-Dimension - Y_max: 1.3650

--- Function 8 ---
/workspaces/codespaces-jupyter/data/initial_data/function_8/initial_inputs.npy
8-Dime